# Databricks Genie via Amazon Bedrock AgentCore Gateway (MCP)

Expose a [Databricks Genie](https://docs.databricks.com/en/genie/index.html) space as a governed MCP tool to Amazon Bedrock agents through [Amazon Bedrock AgentCore Gateway](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway.html). Bedrock agents ask plain-English business questions; Genie returns lakehouse-native SQL answers with Unity Catalog governance, identity, and lineage preserved end-to-end.

![Architecture](images/architecture.png)


## About this sample

This sample was authored by **Antony Prasad Thevaraj**, Senior Specialist Solutions Architect at Databricks, focused on the AWS partnership. It complements the two existing Databricks integrations in this folder:

- [`databricks-dbsql-agentcore-gateway`](../databricks-dbsql-agentcore-gateway) — Databricks SQL MCP via Gateway with M2M auth
- [`databricks-dbsql-per-user-delegation`](../databricks-dbsql-per-user-delegation) — Per-user delegation via RFC 8693 token exchange

The new sample adds the **Genie** surface — Databricks' natural-language analytics layer grounded in Unity Catalog Trusted Assets — so Bedrock agents can ask business questions and receive grounded, governed SQL answers without a custom NL-to-SQL chain.


## Databricks Genie Overview

[Databricks Genie](https://docs.databricks.com/en/genie/index.html) is a natural-language analytics surface grounded in [Unity Catalog](https://docs.databricks.com/en/data-governance/unity-catalog/index.html). It uses Trusted Assets — curated metrics, sample queries, and table descriptions — to ground SQL generation in business semantics. A Genie *space* is the unit you point at a specific data domain (finance, marketing, operations, etc.).

Databricks ships [managed MCP servers](https://docs.databricks.com/en/generative-ai/mcp/managed-mcp.html) that expose Genie spaces via a Model Context Protocol endpoint at `/api/2.0/mcp/genie/{space_id}`. The endpoint exposes a `query_genie` tool that takes a natural-language question and returns the SQL, narrative, and result set.


## Amazon Bedrock AgentCore Gateway Overview

[Amazon Bedrock AgentCore Gateway](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway.html) is a managed MCP gateway in front of your tool surfaces. For this sample it handles:

- **Inbound auth** — agent → gateway authorization (via AgentCore Identity, optionally fronted by Amazon Cognito)
- **Outbound auth** — Databricks OAuth2 M2M credentials, registered via `CreateOauth2CredentialProvider`, retrieved by Gateway at tool-invocation time
- **Routing** — forwards `tools/call` MCP requests to the Databricks-managed Genie MCP endpoint
- **Audit** — emits CloudWatch traces for every tool invocation


## Setting up Databricks Credentials

To authenticate AgentCore Gateway with your Databricks workspace, you need a **service principal** with an OAuth secret. This enables machine-to-machine (M2M) authentication using the OAuth2 client credentials flow.

### Step 1: Create a Service Principal

1. In your Databricks workspace, go to **Settings** → **Identity and access** → **Service principals**
2. Click **Add service principal** and provide a name (e.g., `agentcore-genie-gateway`)
3. After creation, note the **Application ID** — this is your `client_id`

### Step 2: Generate an OAuth Secret

1. Select the service principal you just created
2. Go to the **Secrets** tab and click **Generate secret**
3. Copy the **Secret** value immediately — it will not be shown again. This is your `client_secret`

### Step 3: Grant Genie Space + Unity Catalog Permissions

1. **Genie Space** — open your Genie space → **Settings** → grant the service principal **CAN RUN** on the space
2. **Unity Catalog tables** behind the Genie space — grant `USE CATALOG`, `USE SCHEMA`, `SELECT` to the service principal

For more details, see the [Databricks OAuth M2M documentation](https://docs.databricks.com/en/dev-tools/auth/oauth-m2m.html).


### Databricks Managed Genie MCP Server

Databricks managed MCP servers are available out of the box — no additional setup required beyond authentication. The Genie MCP server endpoint is:

```
https://<your-workspace-host>/api/2.0/mcp/genie/<genie-space-id>
```

Capture the **Genie space ID** from the workspace UI: Genie → your space → URL contains `/spaces/<id>`. For the full list of managed MCP servers, see the [Databricks managed MCP docs](https://docs.databricks.com/en/generative-ai/mcp/managed-mcp.html).


## Step 1: Configure Your Environment

Set your Databricks workspace host, service principal credentials, and Genie space ID. Replace the placeholder values below with your own.


In [ ]:
import os

# Databricks workspace URL (e.g., https://dbc-xxxxxxxx-xxxx.cloud.databricks.com)
DATABRICKS_HOST = ""

# Service principal credentials from the steps above
DATABRICKS_CLIENT_ID = ""      # Application ID of the service principal
DATABRICKS_CLIENT_SECRET = ""  # OAuth secret you generated

# Genie space ID (from the URL in your Databricks workspace)
GENIE_SPACE_ID = ""

# AWS region for AgentCore resources
REGION = "us-east-1"

assert DATABRICKS_HOST,        "Please set DATABRICKS_HOST"
assert DATABRICKS_CLIENT_ID,    "Please set DATABRICKS_CLIENT_ID"
assert DATABRICKS_CLIENT_SECRET,"Please set DATABRICKS_CLIENT_SECRET"
assert GENIE_SPACE_ID,          "Please set GENIE_SPACE_ID"


Install dependencies and create boto3 clients:


In [ ]:
%pip install --quiet boto3 bedrock-agentcore bedrock-agentcore-starter-toolkit


In [ ]:
import json
import time
import logging

import boto3
from bedrock_agentcore_starter_toolkit.operations.gateway.client import GatewayClient

sts = boto3.client('sts')
account_id = sts.get_caller_identity().get('Account')
agentcore = boto3.client('bedrock-agentcore-control', region_name=REGION)

print(f'AWS Account: {account_id}')
print(f'Region:      {REGION}')


## Step 2: Create or Reuse an AgentCore Gateway

Two options:

- **Option A** — Create a new gateway from scratch (run the cells in this section)
- **Option B** — Reuse an existing gateway by loading its configuration from `gateway_config.json`

If you already have a gateway deployed (e.g., from one of the sibling samples), skip to Option B.


### Option A: Create a New Gateway


In [ ]:
client = GatewayClient(region_name=REGION)
client.logger.setLevel(logging.INFO)

print('Creating Cognito authorizer (inbound auth)...')
cognito = client.create_oauth_authorizer_with_cognito('DatabricksGenieGateway')

print('Creating Gateway...')
gateway = client.create_mcp_gateway(
    name='DatabricksGenieGateway',
    role_arn=None,
    authorizer_config=cognito['authorizer_config'],
    enable_semantic_search=True,
)
client.fix_iam_permissions(gateway)

gateway_url = gateway['gatewayUrl']
gateway_id  = gateway['gatewayId']

print(f'Gateway URL: {gateway_url}')
print(f'Gateway ID:  {gateway_id}')
print('Waiting 30s for IAM propagation...')
time.sleep(30)


### Create Databricks OAuth2 Credential Provider (Outbound Auth)

AgentCore Identity manages the outbound OAuth2 credentials so the gateway can authenticate with Databricks on behalf of your agent. We create a credential provider that stores the service principal's client ID and secret, and points at the Databricks OIDC token endpoint.


In [ ]:
host = DATABRICKS_HOST.rstrip('/')
token_endpoint = f'{host}/oidc/v1/token'

print('Creating Databricks OAuth2 credential provider...')
cred_provider = agentcore.create_oauth2_credential_provider(
    name='databricks-genie-oauth',
    credentialProviderVendor='CustomOauth2',
    oauth2ProviderConfigInput={
        'customOauth2ProviderConfig': {
            'oauthDiscovery': {
                'authorizationServerMetadata': {
                    'issuer':                host,
                    'tokenEndpoint':         token_endpoint,
                    'authorizationEndpoint': token_endpoint,
                }
            },
            'clientId':     DATABRICKS_CLIENT_ID,
            'clientSecret': DATABRICKS_CLIENT_SECRET,
        }
    },
)

provider_arn = cred_provider['credentialProviderArn']
secret_arn   = cred_provider.get('secretArn') or cred_provider.get('clientSecretArn', {}).get('secretArn', '')
print(f'Credential provider ARN: {provider_arn}')


### Update Gateway Role Permissions

The gateway's IAM role needs three permissions to use the credential provider end-to-end: fetch workload access tokens, retrieve OAuth2 tokens from the credential provider, and read the stored secret.


In [ ]:
print('Updating gateway role permissions...')
gateway_details = agentcore.get_gateway(gatewayIdentifier=gateway_id)
role_arn  = gateway_details['roleArn']
role_name = role_arn.split('/')[-1]

iam = boto3.client('iam')
policy_doc = json.dumps({
    'Version': '2012-10-17',
    'Statement': [
        {
            'Effect':   'Allow',
            'Action':   'bedrock-agentcore:GetWorkloadAccessToken',
            'Resource': [
                f'arn:aws:bedrock-agentcore:{REGION}:*:workload-identity-directory/default',
                f'arn:aws:bedrock-agentcore:{REGION}:*:workload-identity-directory/default/workload-identity/DatabricksGenieGateway-*',
            ],
        },
        {
            'Effect':   'Allow',
            'Action':   'bedrock-agentcore:GetResourceOauth2Token',
            'Resource': provider_arn,
        },
        {
            'Effect':   'Allow',
            'Action':   'secretsmanager:GetSecretValue',
            'Resource': secret_arn,
        },
    ],
})

iam.put_role_policy(
    RoleName=role_name,
    PolicyName='DatabricksGenieOAuthAccess',
    PolicyDocument=policy_doc,
)
print(f'Updated role: {role_name}')
time.sleep(10)


### Add the Databricks Genie MCP Server as a Gateway Target

Register the Databricks-managed Genie MCP endpoint as a target on the gateway. The gateway will use the OAuth2 credential provider we just created to authenticate outbound requests to Databricks.


In [ ]:
mcp_url = f'{host}/api/2.0/mcp/genie/{GENIE_SPACE_ID}'

print(f'Adding Databricks Genie MCP server target: {mcp_url}')
target = agentcore.create_gateway_target(
    gatewayIdentifier=gateway_id,
    name='DatabricksGenie',
    description=f'Databricks Genie space {GENIE_SPACE_ID} as MCP tool',
    targetConfiguration={
        'mcp': {
            'mcpServer': {
                'endpoint': mcp_url,
            }
        }
    },
    credentialProviderConfigurations=[
        {
            'credentialProviderType': 'OAUTH',
            'credentialProvider': {
                'oauthCredentialProvider': {
                    'providerArn': provider_arn,
                    'grantType':   'CLIENT_CREDENTIALS',
                    'scopes':      ['all-apis'],
                }
            },
        }
    ],
)

target_id = target['targetId']
print(f'Target ID: {target_id}')


Wait for the target to become ready, then synchronize the tool surface from Databricks:


In [ ]:
print('Waiting for target to be ready...')
for _ in range(24):
    t = agentcore.get_gateway_target(gatewayIdentifier=gateway_id, targetId=target_id)
    if t.get('status') not in ('Creating', 'Updating'):
        break
    time.sleep(5)
print(f'Target status: {t.get("status")}')

print('Synchronizing tools from Databricks...')
agentcore.synchronize_gateway_targets(
    gatewayIdentifier=gateway_id,
    targetIdList=[target_id],
)
print('Tools synchronized.')


### Save Configuration

Save the gateway + target identifiers so subsequent runs (or a deployed agent) can reuse them.


In [ ]:
config = {
    'gateway_id':     gateway_id,
    'gateway_url':    gateway_url,
    'target_id':      target_id,
    'provider_arn':   provider_arn,
    'genie_space_id': GENIE_SPACE_ID,
    'region':         REGION,
}
with open('gateway_config.json', 'w') as f:
    json.dump(config, f, indent=2)
print('Saved gateway_config.json')


### Option B: Reuse an Existing Gateway


In [ ]:
import json
with open('gateway_config.json') as f:
    config = json.load(f)
gateway_id     = config['gateway_id']
gateway_url    = config['gateway_url']
target_id      = config['target_id']
provider_arn   = config['provider_arn']
GENIE_SPACE_ID = config['genie_space_id']
print(f'Loaded gateway {gateway_id}, target {target_id}')


## Step 3: Invoke the Genie Tool via a Bedrock Agent

In the Bedrock console (or via the SDK):

1. Open or create a Bedrock agent (Claude, Nova, etc.)
2. Add an action group of type **Gateway** and select `DatabricksGenieGateway`
3. Save and prepare the agent; create or reuse an alias

Then invoke the agent from this notebook. The trace prints reveal when the agent is calling the `query_genie` tool versus answering from its own context.


In [ ]:
import uuid

BEDROCK_AGENT_ID = ''             # Your prepared Bedrock agent ID
BEDROCK_AGENT_ALIAS = 'TSTALIASID'  # or your alias ID

runtime = boto3.client('bedrock-agent-runtime', region_name=REGION)

def ask(question: str) -> str:
    session_id = str(uuid.uuid4())
    resp = runtime.invoke_agent(
        agentId=BEDROCK_AGENT_ID,
        agentAliasId=BEDROCK_AGENT_ALIAS,
        sessionId=session_id,
        inputText=question,
        enableTrace=True,
    )
    chunks: list[str] = []
    for event in resp.get('completion', []):
        if 'chunk' in event:
            chunks.append(event['chunk']['bytes'].decode('utf-8'))
        if 'trace' in event:
            t = event['trace'].get('trace', {})
            if 'orchestrationTrace' in t and 'invocationInput' in t['orchestrationTrace']:
                inv = t['orchestrationTrace']['invocationInput']
                if 'actionGroupInvocationInput' in inv:
                    print(f'[trace] tool call -> ' + inv['actionGroupInvocationInput'].get('apiPath', ''))
    return ''.join(chunks)


## Sample Prompts

Try these prompts against your Bedrock agent. They are intentionally generic so they work against most Genie spaces. Customize them to your domain (finance, marketing, ops, etc.) for sharper answers.


In [ ]:
print(ask('What were our top 5 products by revenue last quarter?'))


In [ ]:
print(ask('How has monthly active users changed over the last 12 months?'))


In [ ]:
print(ask('Break down sales by region and product category for the last fiscal year.'))


In [ ]:
print(ask('Which sales reps exceeded their quota by more than 20 percent in Q3?'))


In [ ]:
print(ask('What does our churn-rate metric measure, and what was it last month?'))


## Validate Governance

- Run the same question as a different Bedrock-authenticated identity — they should get an answer scoped to their Unity Catalog permissions (if you've configured per-user delegation; see the [`databricks-dbsql-per-user-delegation`](../databricks-dbsql-per-user-delegation) sample).
- Inspect Unity Catalog audit logs to confirm the SQL was attributed to the service principal (in M2M mode) or to the end user (in delegation mode).
- Confirm CloudWatch traces capture each `query_genie` invocation with its session ID.


## Troubleshooting

| Symptom | Likely cause |
| --- | --- |
| `401` from Genie endpoint | Service principal lacks `CAN RUN` on the Genie space, or token URL is wrong |
| `403` on Unity Catalog tables | Service principal missing `USE CATALOG` / `USE SCHEMA` / `SELECT` |
| Bedrock agent times out | Genie space too broad — narrow Trusted Assets, or pre-warm with sample questions |
| `query_genie` not visible to the agent | Action group not attached, or `synchronize_gateway_targets` not run after target creation |
| `ParamValidationError` on `create_gateway_target` | Verify `boto3` is current; this sample requires the `mcpServer` shape (recent SDK versions) |


## Clean Up

Tear down the resources in reverse order of creation:


In [ ]:
agentcore.delete_gateway_target(gatewayIdentifier=gateway_id, targetId=target_id)
print('Deleted gateway target.')


In [ ]:
agentcore.delete_oauth2_credential_provider(name='databricks-genie-oauth')
print('Deleted OAuth2 credential provider.')


In [ ]:
agentcore.delete_gateway(gatewayIdentifier=gateway_id)
print('Deleted gateway.')
